# Neural Local Volatility — v7 diagnostic / design-audit notebook

This notebook is deliberately **training-independent**. It does **not** create optimizers, call `backward()`, or update neural-network weights.

Its purpose is to answer the empirical questions that should determine the next v7 revision, rather than adding preventive mechanisms before establishing that the corresponding problem exists.

It reuses the exact v7 March-2024 data cleaning, OTM construction, deterministic 80/20 split, maturity-head architecture and saved v6/v7 checkpoints. It then audits:

1. maturity density and the effective weighting induced by maturity balancing;
2. how much information the 25-quotes-per-slice cap discards;
3. whether raw RMSE and maturity-balanced RMSE actually select different saved epochs;
4. whether the rolling training histories were still improving when they stopped;
5. control-variate quality on every saved calibration date;
6. how much the broad full-chain next-day evaluation inflates errors relative to OTM-matched contracts;
7. Neural-LV delta hedging versus the OptionMetrics market delta on the same contracts;
8. residual concentration by maturity and moneyness;
9. the size of the rolling trunk-time-transfer mismatch and the effect of the exact first-layer time-shift correction.

**Not included:** gradient-clipping activation cannot be recovered retrospectively because the pre-clipping gradient norms were not saved. No short continuation run is performed here, because that would violate the training-independent scope.


### Crash-safety design

This revision is intentionally conservative about memory:

- diagnostics default to **CPU**;
- checkpoint tensor storages use `torch.load(..., mmap=True)` when supported;
- **only one `.pt` checkpoint is opened at a time**;
- Section 4 extracts history/rerank metadata and immediately releases the checkpoint;
- model/CV diagnostics copy only the required final weights into live modules, release the payload, run the diagnostic, then delete the modules;
- no optimizer is ever instantiated and no training state is retained across dates.

The notebook therefore never builds a list of loaded v7 checkpoint payloads, which was the cause of the previous RAM spike.


In [1]:

from pathlib import Path
from dataclasses import dataclass
import gc, json, math, os, re, warnings

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.stats import spearmanr
from tqdm import tqdm
import torch
import torch.nn as nn

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 12345
ANCHOR_DATE = pd.Timestamp("2024-03-01")

# Exact v7 data/filter constants.
SECID = 108105
MIN_DTE, MAX_DTE = 7, 365
MAX_ABS_LOG_MONEYNESS = 0.30
MIN_ABS_DELTA, MAX_ABS_DELTA = 0.05, 0.95
MAX_RELATIVE_SPREAD = 0.25
ATM_TIGHTER_SIDE_BAND = 0.05
MIN_POINTS_PER_SLICE = 8

HELDOUT_FRACTION = 0.20
MAX_QUOTES_PER_SLICE = 25
BASE_STEPS_PER_YEAR = 96

SDE_HIDDEN_LAYERS, SDE_WIDTH = 4, 50
HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH = 3, 30

# Diagnostics only: simulation, never training.
CV_PATHS = 8_192
CV_CONTRACTS = 64
TRANSFER_X_GRID = np.linspace(0.75, 1.25, 81)
TRANSFER_POINTS_PER_INTERVAL = 3

V7_CACHE_DIR = Path("neural_lv_v7_cache")
V6_CACHE_DIR = Path("neural_lv_v6_cache")
DIAG_DIR = Path("neural_lv_v7_diagnostics")
DIAG_DIR.mkdir(exist_ok=True)

# Diagnostics default to CPU: they do not need accelerator memory and this avoids
# retaining large checkpoint tensors on GPU/XPU.
USE_ACCELERATOR = False

def choose_device():
    if USE_ACCELERATOR and torch.cuda.is_available():
        return torch.device("cuda")
    if USE_ACCELERATOR and hasattr(torch, "xpu"):
        try:
            if torch.xpu.is_available():
                return torch.device("xpu")
        except Exception:
            pass
    return torch.device("cpu")

DEVICE = choose_device()
if DEVICE.type == "cpu":
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

def set_seed(seed):
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

set_seed(SEED)
print("Device:", DEVICE)
print("Training-independent diagnostic notebook")
print("v7 cache:", V7_CACHE_DIR.resolve())
print("v6 cache:", V6_CACHE_DIR.resolve())


Device: cpu
Training-independent diagnostic notebook
v7 cache: C:\Users\Startklar\ICL\NDEs\Codes\NSDE\neural_lv_v7_cache
v6 cache: C:\Users\Startklar\ICL\NDEs\Codes\NSDE\neural_lv_v6_cache


## 1. Rebuild the exact v7 March data and daily splits

In [2]:
def find_file(pattern):
    search_dirs = [Path("."), Path("/mnt/data")]
    found = []
    for folder in search_dirs:
        found.extend(sorted(folder.glob(pattern)))
    if not found:
        raise FileNotFoundError(f"Could not find a file matching {pattern!r}")
    return found[0]


def prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw):
    options = options_raw.copy()
    options["date"] = pd.to_datetime(options["date"])
    options["exdate"] = pd.to_datetime(options["exdate"])
    options["K"] = options["strike_price"] / 1000.0
    options["T_days"] = (options["exdate"] - options["date"]).dt.days
    options["T"] = options["T_days"] / 365.0
    options["bid"] = pd.to_numeric(options["best_bid"], errors="coerce")
    options["ask"] = pd.to_numeric(options["best_offer"], errors="coerce")
    options["mid"] = 0.5 * (options["bid"] + options["ask"])
    options["spread"] = options["ask"] - options["bid"]
    options["relative_spread"] = options["spread"] / options["mid"].replace(0, np.nan)
    options["iv_market"] = pd.to_numeric(options["impl_volatility"], errors="coerce")
    options["delta_market"] = pd.to_numeric(options["delta"], errors="coerce")
    options["abs_delta_market"] = options["delta_market"].abs()

    mask = (
        (options["secid"] == SECID)
        & (options["exercise_style"] == "E")
        & (options["contract_size"] == 100)
        & (options["cfadj"] == 1)
        & options["cp_flag"].isin(["C", "P"])
        & (options["bid"] >= 0)
        & (options["ask"] > options["bid"])
        & (options["mid"] > 0)
        & options["T_days"].between(MIN_DTE, MAX_DTE)
        & options["iv_market"].between(1e-4, 5.0)
        & options["abs_delta_market"].between(MIN_ABS_DELTA, MAX_ABS_DELTA)
        & (options["relative_spread"] <= MAX_RELATIVE_SPREAD)
    )
    options = options.loc[mask].copy()

    spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
    spot["date"] = pd.to_datetime(spot["date"])
    spot = spot.rename(columns={"close": "S"}).drop_duplicates("date")

    forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
    forwards["date"] = pd.to_datetime(forwards["date"])
    forwards["expiration"] = pd.to_datetime(forwards["expiration"])
    forwards = forwards.rename(columns={
        "expiration": "exdate",
        "AMSettlement": "am_settlement",
        "ForwardPrice": "F",
    })
    forwards = forwards[
        ["date", "exdate", "am_settlement", "F"]
    ].drop_duplicates(["date", "exdate", "am_settlement"])

    rates = rates_raw.copy()
    rates["date"] = pd.to_datetime(rates["date"])
    rates["days"] = pd.to_numeric(rates["days"], errors="coerce")
    rates["rate_decimal"] = pd.to_numeric(rates["rate"], errors="coerce") / 100.0
    rate_curves = {
        d: (g["days"].to_numpy(float), g["rate_decimal"].to_numpy(float))
        for d, g in rates.dropna().sort_values(["date", "days"]).groupby("date")
    }

    options = options.merge(spot, on="date", how="left").merge(
        forwards,
        on=["date", "exdate", "am_settlement"],
        how="left",
        validate="many_to_one",
    )
    options["r"] = [
        float(np.interp(n, *rate_curves[d])) if d in rate_curves else np.nan
        for d, n in zip(options["date"], options["T_days"])
    ]
    options["D"] = np.exp(-options["r"] * options["T"])
    options["carry"] = np.log(options["F"] / options["S"]) / options["T"]
    options["q"] = options["r"] - options["carry"]
    options["k"] = np.log(options["K"] / options["F"])

    options = options.dropna(subset=["S", "F", "r", "D", "carry", "k"])
    options = options.loc[options["k"].abs() <= MAX_ABS_LOG_MONEYNESS].copy()

    call_lb = options["D"] * np.maximum(options["F"] - options["K"], 0.0)
    call_ub = options["D"] * options["F"]
    put_lb = options["D"] * np.maximum(options["K"] - options["F"], 0.0)
    put_ub = options["D"] * options["K"]
    lb = np.where(options["cp_flag"].eq("C"), call_lb, put_lb)
    ub = np.where(options["cp_flag"].eq("C"), call_ub, put_ub)
    options = options.loc[
        (options["mid"] >= lb - 1e-6) & (options["mid"] <= ub + 1e-6)
    ].copy()

    return options.sort_values(
        ["date", "exdate", "am_settlement", "K", "cp_flag"]
    ).reset_index(drop=True)


def build_otm_surface(quotes):
    idx = [
        "date", "exdate", "am_settlement", "K", "T", "S", "F",
        "r", "q", "carry", "D", "k"
    ]
    vals = [
        "mid", "bid", "ask", "spread", "relative_spread",
        "iv_market", "delta_market", "optionid"
    ]

    wide = quotes.pivot_table(
        index=idx, columns="cp_flag", values=vals, aggfunc="first"
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    strict_call = wide["K"] >= wide["F"]
    near = wide["k"].abs() <= ATM_TIGHTER_SIDE_BAND
    have_c = wide.get("iv_market_C", pd.Series(np.nan, index=wide.index)).notna()
    have_p = wide.get("iv_market_P", pd.Series(np.nan, index=wide.index)).notna()
    c_spread = wide.get("spread_C", pd.Series(np.inf, index=wide.index))
    p_spread = wide.get("spread_P", pd.Series(np.inf, index=wide.index))

    choose_call = strict_call.copy()
    both_near = near & have_c & have_p
    choose_call = np.where(both_near, c_spread <= p_spread, choose_call)
    wide["cp_otm"] = np.where(choose_call, "C", "P")

    for base in vals:
        c = wide.get(f"{base}_C", pd.Series(np.nan, index=wide.index))
        p = wide.get(f"{base}_P", pd.Series(np.nan, index=wide.index))
        wide[f"{base}_otm"] = np.where(choose_call, c, p)

    wide = wide.dropna(subset=["iv_market_otm", "mid_otm", "optionid_otm"]).copy()

    sizes = wide.groupby(["date", "exdate", "am_settlement"]).size()
    valid = sizes[sizes >= MIN_POINTS_PER_SLICE].index

    return (
        wide.set_index(["date", "exdate", "am_settlement"])
        .loc[valid]
        .reset_index()
        .sort_values(["date", "T", "k"])
        .reset_index(drop=True)
    )

FILES = {
    "options": find_file("one_month_spx_option_price.csv"),
    "forwards": find_file("one_month_spx_forward_price.csv"),
    "spot": find_file("one_month_spx_security_price.csv"),
    "rates": find_file("one_month_spx_zero_coupon_yield_curve.csv"),
}

options_raw = pd.read_csv(FILES["options"], low_memory=False)
forwards_raw = pd.read_csv(FILES["forwards"], low_memory=False)
spot_raw = pd.read_csv(FILES["spot"], low_memory=False)
rates_raw = pd.read_csv(FILES["rates"], low_memory=False)

quotes_all = prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw)
otm = build_otm_surface(quotes_all)

print("Complete cleaned quotes:", len(quotes_all))
print("OTM observations:", len(otm))
print("Trading dates:", otm["date"].nunique())


Complete cleaned quotes: 192031
OTM observations: 91930
Trading dates: 20


In [3]:
class DailyMarketCurve:
    date: pd.Timestamp
    am_settlement: int
    spot: float
    T_nodes: np.ndarray
    logD_nodes: np.ndarray
    logF_nodes: np.ndarray

    def __post_init__(self):
        self._logD = PchipInterpolator(self.T_nodes, self.logD_nodes, extrapolate=True)
        self._logF = PchipInterpolator(self.T_nodes, self.logF_nodes, extrapolate=True)

    def log_discount(self, t):
        return self._logD(np.maximum(np.asarray(t, float), 0.0))

    def log_forward_ratio(self, t):
        return self._logF(np.maximum(np.asarray(t, float), 0.0))

    def discount(self, t0, t1):
        return np.exp(self.log_discount(t1) - self.log_discount(t0))

    def forward(self, t):
        return self.spot * np.exp(self.log_forward_ratio(t))


class CurveBook:
    def __init__(self, spot_raw, forwards_raw, rates_raw):
        spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
        spot["date"] = pd.to_datetime(spot["date"])
        self.spot = spot.set_index("date")["close"].to_dict()

        self.forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
        self.forwards["date"] = pd.to_datetime(self.forwards["date"])
        self.forwards["expiration"] = pd.to_datetime(self.forwards["expiration"])

        self.rates = rates_raw.copy()
        self.rates["date"] = pd.to_datetime(self.rates["date"])
        self.rates["rate_decimal"] = pd.to_numeric(self.rates["rate"], errors="coerce") / 100.0
        self._cache = {}

    def get(self, date, am_settlement):
        date, am = pd.Timestamp(date), int(am_settlement)
        key = (date, am)
        if key in self._cache:
            return self._cache[key]

        S0 = float(self.spot[date])
        rg = self.rates.loc[self.rates["date"] == date, ["days", "rate_decimal"]].dropna().sort_values("days")
        T_r = rg["days"].to_numpy(float) / 365.0
        logD = -rg["rate_decimal"].to_numpy(float) * T_r

        fg = self.forwards.loc[
            (self.forwards["date"] == date) & (self.forwards["AMSettlement"] == am),
            ["expiration", "ForwardPrice"],
        ].copy()
        if len(fg) < 2:
            fg = self.forwards.loc[self.forwards["date"] == date, ["expiration", "ForwardPrice"]].copy()

        fg["T"] = (fg["expiration"] - date).dt.days / 365.0
        fg = (
            fg.loc[(fg["T"] >= 0) & (fg["ForwardPrice"] > 0)]
            .groupby("T", as_index=False)["ForwardPrice"].mean().sort_values("T")
        )
        T_f = fg["T"].to_numpy(float)
        logF = np.log(fg["ForwardPrice"].to_numpy(float) / S0)
        positive = T_f > 0
        T_f, logF = T_f[positive], logF[positive]

        T_all = np.unique(np.concatenate([[0.0], T_r, T_f]))
        logD_all = np.interp(T_all, np.r_[0.0, T_r], np.r_[0.0, logD])
        logF_all = np.interp(T_all, np.r_[0.0, T_f], np.r_[0.0, logF])

        curve = DailyMarketCurve(date, am, S0, T_all, logD_all, logF_all)
        self._cache[key] = curve
        return curve


def sample_calibration_day(day, max_per_slice=MAX_QUOTES_PER_SLICE):
    pieces = []
    for _, g in day.groupby(["exdate", "am_settlement"]):
        g = g.sort_values("k")
        if len(g) <= max_per_slice:
            pieces.append(g)
        else:
            pos = np.unique(np.round(np.linspace(0, len(g)-1, max_per_slice)).astype(int))
            pieces.append(g.iloc[pos])
    return pd.concat(pieces, ignore_index=True)


def prepare_selected_universe(date):
    day = otm.loc[otm["date"] == pd.Timestamp(date)].copy()
    selected = sample_calibration_day(day).rename(columns={
        "mid_otm": "target_mid",
        "cp_otm": "target_cp",
        "optionid_otm": "optionid",
    }).reset_index(drop=True)
    selected["full_selected_id"] = np.arange(len(selected), dtype=int)
    return selected


def stratified_holdout(selected, fraction=HELDOUT_FRACTION, seed=SEED+77):
    """~80/20 within each expiry/settlement slice, spread across moneyness."""
    selected = selected.copy().reset_index(drop=True)
    is_holdout = np.zeros(len(selected), dtype=bool)
    rng = np.random.default_rng(seed)
    stride = max(2, int(round(1.0 / fraction)))

    for _, g in selected.groupby(["exdate", "am_settlement"], sort=False):
        ordered = g.sort_values("k").index.to_numpy()
        chosen = ordered[int(rng.integers(0, stride))::stride]
        if len(chosen) == 0:
            chosen = ordered[[len(ordered)//2]]
        if len(chosen) >= len(ordered):
            chosen = chosen[:-1]
        is_holdout[chosen] = True

    selected["split"] = np.where(is_holdout, "holdout", "train")
    train = selected.loc[~is_holdout].copy().reset_index(drop=True)
    holdout = selected.loc[is_holdout].copy().reset_index(drop=True)
    train["contract_id"] = np.arange(len(train), dtype=int)
    return selected, train, holdout


def daily_split(date):
    selected = prepare_selected_universe(date)
    return stratified_holdout(selected)

curve_book = CurveBook(spot_raw, forwards_raw, rates_raw)
trading_dates = sorted(pd.to_datetime(quotes_all["date"].unique()))

anchor_selected, anchor_train, anchor_holdout = daily_split(ANCHOR_DATE)
print("Anchor selected/train/holdout:", len(anchor_selected), len(anchor_train), len(anchor_holdout))


Anchor selected/train/holdout: 997 798 199


## 2. Maturity-density and 25-quote-cap audit

Two separate questions are measured here:

- **Does the March dataset actually have materially unequal maturity density after the current sampling/split?**
- **How much of the filtered OTM surface is the 25-per-slice cap discarding?**

For each training date, `balanced/raw weight ratio` is the per-contract weight under the current maturity-balanced loss divided by its weight under ordinary raw contract-level MSE:

\[

rac{1/(M n_m)}{1/N}=
rac{N}{M n_m}.
\]

A ratio near 1 for every maturity means maturity balancing is doing almost nothing. Large dispersion means it materially reweights the dataset.


In [4]:

def maturity_and_cap_audit(date):
    date = pd.Timestamp(date)
    day = otm.loc[otm["date"] == date].copy()

    # Before the 25-quote cap.
    full_slice = (
        day.groupby(["exdate", "am_settlement"])
        .agg(
            available=("k", "size"),
            k_min_full=("k", "min"),
            k_max_full=("k", "max"),
        )
        .reset_index()
    )

    selected, train, holdout = daily_split(date)
    sel_slice = (
        selected.groupby(["exdate", "am_settlement"])
        .agg(
            selected=("k", "size"),
            k_min_selected=("k", "min"),
            k_max_selected=("k", "max"),
        )
        .reset_index()
    )
    slice_table = full_slice.merge(sel_slice, on=["exdate", "am_settlement"], how="left")
    slice_table["retained_fraction"] = slice_table["selected"] / slice_table["available"]
    full_span = (slice_table["k_max_full"] - slice_table["k_min_full"]).replace(0, np.nan)
    selected_span = slice_table["k_max_selected"] - slice_table["k_min_selected"]
    slice_table["moneyness_span_retained"] = selected_span / full_span
    slice_table["date"] = date

    # Effective contract weights under current maturity-balanced objective.
    counts = train.groupby("T").size().rename("n_train").reset_index()
    N, M = len(train), len(counts)
    counts["raw_contract_weight"] = 1.0 / N
    counts["balanced_contract_weight"] = 1.0 / (M * counts["n_train"])
    counts["balanced_over_raw"] = counts["balanced_contract_weight"] / counts["raw_contract_weight"]
    counts["date"] = date

    summary = {
        "date": date,
        "n_otm_available": len(day),
        "n_selected": len(selected),
        "n_train": len(train),
        "n_holdout": len(holdout),
        "n_maturities": M,
        "train_per_maturity_min": int(counts["n_train"].min()),
        "train_per_maturity_median": float(counts["n_train"].median()),
        "train_per_maturity_max": int(counts["n_train"].max()),
        "train_per_maturity_CV": float(counts["n_train"].std(ddof=0) / counts["n_train"].mean()),
        "weight_ratio_min": float(counts["balanced_over_raw"].min()),
        "weight_ratio_median": float(counts["balanced_over_raw"].median()),
        "weight_ratio_max": float(counts["balanced_over_raw"].max()),
        "overall_selected_fraction": float(len(selected) / len(day)),
        "median_slice_retained_fraction": float(slice_table["retained_fraction"].median()),
        "median_moneyness_span_retained": float(slice_table["moneyness_span_retained"].median()),
    }
    return summary, counts, slice_table

density_rows, weight_tables, cap_tables = [], [], []
for d in trading_dates:
    s, w, c = maturity_and_cap_audit(d)
    density_rows.append(s)
    weight_tables.append(w)
    cap_tables.append(c)

density_summary = pd.DataFrame(density_rows)
effective_weights = pd.concat(weight_tables, ignore_index=True)
cap_audit = pd.concat(cap_tables, ignore_index=True)

print("\nMARCH MATURITY / CAP SUMMARY")
print(density_summary.to_string(index=False))

print("\nOverall selected-data fraction:", round(density_summary["n_selected"].sum() / density_summary["n_otm_available"].sum(), 4))
print("Largest balanced/raw per-contract weight ratio:", round(effective_weights["balanced_over_raw"].max(), 4))
print("Smallest balanced/raw per-contract weight ratio:", round(effective_weights["balanced_over_raw"].min(), 4))

density_summary.to_csv(DIAG_DIR / "01_maturity_density_summary.csv", index=False)
effective_weights.to_csv(DIAG_DIR / "01_effective_maturity_weights.csv", index=False)
cap_audit.to_csv(DIAG_DIR / "02_quote_cap_audit.csv", index=False)



MARCH MATURITY / CAP SUMMARY
      date  n_otm_available  n_selected  n_train  n_holdout  n_maturities  train_per_maturity_min  train_per_maturity_median  train_per_maturity_max  train_per_maturity_CV  weight_ratio_min  weight_ratio_median  weight_ratio_max  overall_selected_fraction  median_slice_retained_fraction  median_moneyness_span_retained
2024-03-01             4304         997      798        199            36                      18                       20.0                      40               0.284823          0.554167             1.108333          1.231481                   0.231645                        0.303041                             1.0
2024-03-04             4502        1000      800        200            36                      20                       20.0                      40               0.282843          0.555556             1.111111          1.111111                   0.222124                        0.280934                             1.0
2024-03-05

## 3. Saved-training-history audit: raw RMSE versus maturity-balanced RMSE

No model is retrained. The completed v7 rolling checkpoints already contain the monitoring history and the independent reranking records.

This section asks:

- Did raw RMSE and maturity-balanced RMSE prefer the same monitored epoch?
- How correlated were their rankings?
- What raw-RMSE penalty, if any, came from selecting by the balanced criterion?
- Was raw RMSE still trending down near the stopping point?

Only epochs that were actually monitored in the original v7 can be compared; the original notebook did not save a model/monitor value after every SDE update.


## 3. Saved model definitions and non-training simulation helpers

In [5]:
class FeedForwardTrunk(nn.Module):
    def __init__(self, input_dim=2, hidden_layers=SDE_HIDDEN_LAYERS, width=SDE_WIDTH):
        super().__init__()
        layers, d = [], input_dim
        for _ in range(hidden_layers):
            layers += [nn.Linear(d, width), nn.ReLU()]
            d = width
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def sabate_weight_initialisation(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight.data, gain=1.5)


def _tx(t, x):
    if x.ndim == 1:
        x = x[:, None]
    if not torch.is_tensor(t):
        t = torch.full_like(x, float(t))
    elif t.ndim == 0:
        t = torch.full_like(x, float(t.item()))
    elif t.ndim == 1:
        t = t[:, None]
    return torch.cat([t, x], dim=1)


class MaturityHeadNeuralLocalVol(nn.Module):
    def __init__(self, maturity_boundaries):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.trunk = FeedForwardTrunk()
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(SDE_WIDTH, 1), nn.Softplus())
            for _ in self.boundaries
        ])
        self.apply(sabate_weight_initialisation)

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.heads)-1)

    def forward(self, t, x):
        return self.heads[self.interval_index(t)](self.trunk(_tx(t, x)))


class PerOptionHedgeNet(nn.Module):
    def __init__(self, n_options):
        super().__init__()
        self.trunk = FeedForwardTrunk(2, HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH)
        self.output = nn.Linear(HEDGE_WIDTH, int(n_options))
        self.apply(sabate_weight_initialisation)

    def forward_ids(self, t, x, option_ids):
        h = self.trunk(_tx(t, x))
        ids = torch.as_tensor(option_ids, dtype=torch.long, device=h.device)
        return h @ self.output.weight[ids].T + self.output.bias[ids][None, :]


class IntervalHedgeBank(nn.Module):
    def __init__(self, maturity_boundaries, n_options):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.nets = nn.ModuleList([
            PerOptionHedgeNet(n_options) for _ in self.boundaries
        ])

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.nets)-1)

    def forward_ids(self, t, x, option_ids):
        return self.nets[self.interval_index(t)].forward_ids(t, x, option_ids)


def safe_torch_load(path, map_location="cpu"):
    """Load a checkpoint with tensor storages memory-mapped when supported.

    mmap=True avoids eagerly materialising every optimizer/model tensor in RAM.
    The fallback keeps compatibility with older PyTorch versions.
    """
    kwargs = dict(map_location=map_location)
    try:
        return torch.load(path, mmap=True, weights_only=False, **kwargs)
    except TypeError:
        try:
            return torch.load(path, weights_only=False, **kwargs)
        except TypeError:
            return torch.load(path, **kwargs)


def build_time_grid(maturities, steps_per_year=BASE_STEPS_PER_YEAR):
    maturities = np.sort(np.unique(np.asarray(maturities, float)))
    max_T = float(maturities[-1])
    base = np.arange(0.0, max_T + 0.5/steps_per_year, 1.0/steps_per_year)
    grid = np.unique(np.r_[0.0, base[base <= max_T + 1e-12], maturities, max_T])
    grid.sort()
    return grid


def nearest_grid_indices(grid, times):
    grid, times = np.asarray(grid), np.asarray(times)
    idx = np.clip(np.searchsorted(grid, times), 0, len(grid)-1)
    left = np.maximum(idx-1, 0)
    idx = np.where(np.abs(grid[left]-times) < np.abs(grid[idx]-times), left, idx)
    if np.max(np.abs(grid[idx]-times)) > 1e-10:
        raise RuntimeError("A listed maturity is not represented exactly on the simulation grid.")
    return idx


class CalibrationContext:
    date: pd.Timestamp
    spot: float
    contracts: pd.DataFrame
    maturity_boundaries: np.ndarray
    grid_np: np.ndarray
    grid: torch.Tensor
    maturity_index: torch.Tensor
    stage_index: np.ndarray
    stage_tensor: torch.Tensor
    K: torch.Tensor
    F_T: torch.Tensor
    D_T: torch.Tensor
    target: torch.Tensor
    cp_sign: torch.Tensor
    am: torch.Tensor
    F_grid: torch.Tensor
    D_grid: torch.Tensor


def build_calibration_context(date, contracts):
    date = pd.Timestamp(date)
    contracts = contracts.copy().reset_index(drop=True)
    contracts["contract_id"] = np.arange(len(contracts), dtype=int)

    boundaries = np.sort(contracts["T"].unique())
    grid_np = build_time_grid(boundaries)
    maturity_idx = nearest_grid_indices(grid_np, contracts["T"].to_numpy(float))
    stage_map = {round(float(T), 12): i for i, T in enumerate(boundaries)}
    stage_index = np.array([stage_map[round(float(T), 12)] for T in contracts["T"]], dtype=int)

    F_grid = np.empty((2, len(grid_np)))
    D_grid = np.empty((2, len(grid_np)))
    for am in [0, 1]:
        curve = curve_book.get(date, am)
        F_grid[am] = curve.forward(grid_np)
        D_grid[am] = curve.discount(0.0, grid_np)

    return CalibrationContext(
        date=date,
        spot=float(contracts["S"].iloc[0]),
        contracts=contracts,
        maturity_boundaries=boundaries,
        grid_np=grid_np,
        grid=torch.tensor(grid_np, dtype=torch.float32, device=DEVICE),
        maturity_index=torch.tensor(maturity_idx, dtype=torch.long, device=DEVICE),
        stage_index=stage_index,
        stage_tensor=torch.tensor(stage_index, dtype=torch.long, device=DEVICE),
        K=torch.tensor(contracts["K"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        F_T=torch.tensor(contracts["F"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        D_T=torch.tensor(contracts["D"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        target=torch.tensor(contracts["target_mid"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        cp_sign=torch.tensor(np.where(contracts["target_cp"].eq("C"), 1.0, -1.0), dtype=torch.float32, device=DEVICE),
        am=torch.tensor(contracts["am_settlement"].to_numpy(int), dtype=torch.long, device=DEVICE),
        F_grid=torch.tensor(F_grid, dtype=torch.float32, device=DEVICE),
        D_grid=torch.tensor(D_grid, dtype=torch.float32, device=DEVICE),
    )


def antithetic_normals(total_paths, n_steps, seed):
    half = (total_paths + 1) // 2
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed))
    z = torch.randn(half, n_steps, generator=gen, dtype=torch.float32)
    return torch.cat([z, -z], dim=0)[:total_paths].to(DEVICE)


def simulate_estimators(model, hedge, ctx, contract_ids, z, phase):
    ids_np = np.asarray(contract_ids, dtype=int)
    ids = torch.as_tensor(ids_np, dtype=torch.long, device=DEVICE)
    maturity_idx = ctx.maturity_index[ids]
    K, F_T, D_T = ctx.K[ids], ctx.F_T[ids], ctx.D_T[ids]
    cp, am = ctx.cp_sign[ids], ctx.am[ids]

    B, M = z.shape[0], len(ids_np)
    x = torch.ones((B, 1), dtype=torch.float32, device=DEVICE)
    cv = torch.zeros((B, M), dtype=torch.float32, device=DEVICE)
    raw = torch.zeros((B, M), dtype=torch.float32, device=DEVICE)

    for step in range(int(maturity_idx.max().item())):
        t = ctx.grid[step]
        dt = ctx.grid[step+1] - ctx.grid[step]
        sqrt_dt = torch.sqrt(dt)
        sigma = model(t, x) if phase == "sde" else model(t, x).detach()
        dW = sqrt_dt * z[:, step:step+1]

        active = torch.where(maturity_idx > step)[0]
        if len(active):
            active_ids = ids_np[active.detach().cpu().numpy()]
            h = hedge.forward_ids(t, x.detach(), active_ids)
            if phase == "sde":
                h = h.detach()
            stochastic_piece = x.detach() * sigma.detach() * dW
            discounted_forward = ctx.D_grid[am[active], step] * ctx.F_grid[am[active], step]
            cv[:, active] += h * stochastic_piece * discounted_forward[None, :]

        a = x * sigma
        x = x + a*dW / (1.0 + torch.abs(x.detach()*sigma.detach())*sqrt_dt)

        matured = torch.where(maturity_idx == step+1)[0]
        if len(matured):
            S_T = x * F_T[matured][None, :]
            strike = K[matured][None, :]
            call = torch.clamp(S_T-strike, min=0.0)
            put = torch.clamp(strike-S_T, min=0.0)
            raw[:, matured] = torch.where((cp[matured] > 0)[None, :], call, put) * D_T[matured][None, :]

    return raw, raw-cv


def choose_even_ids(n_total, n):
    if n is None or n >= n_total:
        return np.arange(n_total, dtype=int)
    return np.unique(np.round(np.linspace(0, n_total-1, n)).astype(int))

@torch.no_grad()
def cv_variance_stats(model, hedge, ctx, n_contracts=CV_CONTRACTS, paths=CV_PATHS, seed=SEED+9000):
    ids = choose_even_ids(len(ctx.contracts), n_contracts)
    z = antithetic_normals(paths, len(ctx.grid_np)-1, seed)
    raw, cv = simulate_estimators(model, hedge, ctx, ids, z, phase="sde")
    ratios = (
        cv.var(0, unbiased=False)
        / torch.clamp(raw.var(0, unbiased=False), min=1e-12)
    ).detach().cpu().numpy()
    del z, raw, cv
    gc.collect()
    return {
        "median": float(np.median(ratios)),
        "p90": float(np.quantile(ratios, 0.90)),
        "mean": float(np.mean(ratios)),
        "max": float(np.max(ratios)),
    }


def locate_v6_anchor(date=ANCHOR_DATE):
    stamp = f"{pd.Timestamp(date):%Y%m%d}"
    exact = V6_CACHE_DIR / f"real_{stamp}_split80_20_ablation_rawloss_maturity_heads_resume.pt"
    if exact.exists():
        return exact
    matches = sorted(V6_CACHE_DIR.glob(f"*{stamp}*ablation_rawloss*maturity_heads*resume.pt"))
    return matches[0] if matches else None

def rolling_checkpoint(date):
    return V7_CACHE_DIR / f"rolling_{pd.Timestamp(date):%Y%m%d}_raw_heads_resume.pt"

def checkpoint_path_for_date(date):
    date = pd.Timestamp(date)
    return locate_v6_anchor(date) if date == ANCHOR_DATE else rolling_checkpoint(date)

def available_saved_dates():
    dates = [ANCHOR_DATE] if locate_v6_anchor(ANCHOR_DATE) is not None else []
    for p in sorted(V7_CACHE_DIR.glob("rolling_*_raw_heads_resume.pt")):
        m = re.search(r"rolling_(\d{8})_raw_heads", p.name)
        if m:
            dates.append(pd.to_datetime(m.group(1), format="%Y%m%d"))
    return sorted(pd.unique(dates))

def checkpoint_metadata(path):
    """Extract only small Python metadata, then immediately release the checkpoint."""
    payload = safe_torch_load(path, map_location="cpu")
    try:
        meta = {
            "completed": bool(payload.get("completed", False)),
            "training_finished": bool(payload.get("training_finished", False)),
            "stop_reason": payload.get("stop_reason"),
            "history": list(payload.get("history", [])),
            "rerank_records": list(payload.get("rerank_records") or []),
            "boundaries": np.asarray(payload.get("boundaries", []), float).copy(),
        }
    finally:
        del payload
        gc.collect()
    return meta

def load_saved_model(date, ctx):
    """Load only the final Neural-LV weights into a model, then release the payload."""
    date = pd.Timestamp(date)
    path = checkpoint_path_for_date(date)
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f"No saved checkpoint found for {date.date()}")

    payload = safe_torch_load(path, map_location="cpu")
    try:
        state = payload.get("final_model_state") or payload.get("model_state")
        boundaries = np.asarray(payload.get("boundaries", ctx.maturity_boundaries), float).copy()
        model = MaturityHeadNeuralLocalVol(boundaries).to(DEVICE)
        model.load_state_dict(state, strict=True)
        model.eval()
    finally:
        del payload
        gc.collect()
    return model, Path(path)

def load_saved_model_and_hedge(date, ctx):
    """Load final model+hedge weights one checkpoint at a time.

    Optimizer states remain inside the memory-mapped payload and are released immediately
    after the two state_dicts have been copied into live modules.
    """
    date = pd.Timestamp(date)
    path = checkpoint_path_for_date(date)
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f"No saved checkpoint found for {date.date()}")

    payload = safe_torch_load(path, map_location="cpu")
    try:
        model_state = payload.get("final_model_state") or payload.get("model_state")
        hedge_state = payload.get("final_hedge_state") or payload.get("hedge_state")
        boundaries = np.asarray(payload.get("boundaries", ctx.maturity_boundaries), float).copy()

        model = MaturityHeadNeuralLocalVol(boundaries).to(DEVICE)
        hedge = IntervalHedgeBank(boundaries, len(ctx.contracts)).to(DEVICE)
        model.load_state_dict(model_state, strict=True)
        hedge.load_state_dict(hedge_state, strict=True)
        model.eval()
        hedge.eval()
    finally:
        del payload
        gc.collect()

    return model, hedge, Path(path)


## 4. Raw-versus-balanced checkpoint-selection audit

In [6]:

def parse_date_from_name(path):
    m = re.search(r"rolling_(\d{8})_raw_heads", path.name)
    return pd.to_datetime(m.group(1), format="%Y%m%d") if m else None

checkpoint_paths = sorted(V7_CACHE_DIR.glob("rolling_*_raw_heads_resume.pt"))

history_rows = []
rerank_rows = []
history_csv_parts = []

for path in checkpoint_paths:
    date = parse_date_from_name(path)
    if date is None:
        continue

    # IMPORTANT: exactly one resume checkpoint is mapped/loaded at a time.
    meta = checkpoint_metadata(path)
    if not meta["completed"]:
        del meta
        gc.collect()
        continue

    hist = pd.DataFrame(meta["history"])
    rr = pd.DataFrame(meta["rerank_records"])
    stop_reason = meta["stop_reason"]
    del meta
    gc.collect()

    if len(hist):
        hist["date"] = date
        # Persist each history immediately rather than retaining all histories in RAM.
        per_date_history = DIAG_DIR / f"03_history_{date:%Y%m%d}.csv"
        hist.to_csv(per_date_history, index=False)
        history_csv_parts.append(per_date_history)

        mon = hist.dropna(subset=["objective_rmse", "raw_rmse"]).copy()
        if len(mon):
            balanced_i = mon["objective_rmse"].astype(float).idxmin()
            raw_i = mon["raw_rmse"].astype(float).idxmin()
            b = mon.loc[balanced_i]
            r = mon.loc[raw_i]

            if len(mon) > 1:
                rho = spearmanr(
                    mon["objective_rmse"].astype(float),
                    mon["raw_rmse"].astype(float)
                ).statistic
            else:
                rho = np.nan

            last3 = mon.tail(min(3, len(mon)))
            slope = (
                float(np.polyfit(last3["epoch"].astype(float), last3["raw_rmse"].astype(float), 1)[0])
                if len(last3) >= 2 else np.nan
            )

            history_rows.append({
                "date": date,
                "checkpoint": path.name,
                "n_monitors": len(mon),
                "balanced_best_epoch": int(b["epoch"]),
                "raw_best_epoch": int(r["epoch"]),
                "same_monitor_winner": int(b["epoch"]) == int(r["epoch"]),
                "spearman_balanced_vs_raw": float(rho) if np.isfinite(rho) else np.nan,
                "raw_RMSE_at_balanced_winner": float(b["raw_rmse"]),
                "best_monitored_raw_RMSE": float(r["raw_rmse"]),
                "raw_selection_penalty_pct": float((b["raw_rmse"]/r["raw_rmse"] - 1.0)*100.0),
                "last_monitored_epoch": int(mon["epoch"].iloc[-1]),
                "last_monitored_raw_RMSE": float(mon["raw_rmse"].iloc[-1]),
                "last3_raw_RMSE_slope_per_epoch": slope,
                "stop_reason": stop_reason,
            })

    if len(rr):
        obj_best = rr.loc[rr["rerank_objective_rmse"].astype(float).idxmin()]
        raw_best = rr.loc[rr["rerank_raw_rmse"].astype(float).idxmin()]
        rerank_rows.append({
            "date": date,
            "n_reranked": len(rr),
            "balanced_rerank_epoch": int(obj_best["epoch"]),
            "raw_rerank_epoch": int(raw_best["epoch"]),
            "same_rerank_winner": int(obj_best["epoch"]) == int(raw_best["epoch"]),
            "raw_RMSE_at_balanced_rerank_winner": float(obj_best["rerank_raw_rmse"]),
            "best_rerank_raw_RMSE": float(raw_best["rerank_raw_rmse"]),
            "raw_rerank_selection_penalty_pct": float(
                (obj_best["rerank_raw_rmse"]/raw_best["rerank_raw_rmse"] - 1.0)*100.0
            ),
        })

    del hist, rr
    if "mon" in locals():
        del mon
    gc.collect()

history_audit = pd.DataFrame(history_rows)
rerank_audit = pd.DataFrame(rerank_rows)

print("\nMONITOR-HISTORY AUDIT")
print(history_audit.to_string(index=False) if len(history_audit) else "No completed rolling checkpoints found.")

print("\n50K RERANK AUDIT (only candidates retained by original v7)")
print(rerank_audit.to_string(index=False) if len(rerank_audit) else "No rerank records found.")

if len(history_audit):
    print("\nRaw/balanced monitor winner disagreement rate:",
          round(1.0 - history_audit["same_monitor_winner"].mean(), 3))
if len(rerank_audit):
    print("Raw/balanced rerank winner disagreement rate:",
          round(1.0 - rerank_audit["same_rerank_winner"].mean(), 3))

history_audit.to_csv(DIAG_DIR / "03_checkpoint_history_audit.csv", index=False)
rerank_audit.to_csv(DIAG_DIR / "03_rerank_metric_audit.csv", index=False)

# Reassemble the tiny histories from CSV only if desired; this never retains checkpoint payloads.
if history_csv_parts:
    pd.concat([pd.read_csv(p) for p in history_csv_parts], ignore_index=True).to_csv(
        DIAG_DIR / "03_training_history_long.csv", index=False
    )

del history_rows, rerank_rows
gc.collect()


: 

: 

## 5. Control-variate quality on every saved calibration date

This is forward simulation only—no optimization. A variance ratio of 0.04 means the control-variate estimator retains about 4% of the raw Monte Carlo variance for the sampled contracts.


In [6]:

saved_dates = available_saved_dates()

cv_rows = []
for j, date in enumerate(saved_dates):
    model = hedge = ctx = None
    try:
        _, train, _ = daily_split(date)
        ctx = build_calibration_context(date, train)
        model, hedge, path = load_saved_model_and_hedge(date, ctx)
        stats = cv_variance_stats(model, hedge, ctx, seed=SEED+9000+j*100)
        cv_rows.append({
            "date": date,
            "checkpoint": path.name,
            **{f"cv_ratio_{k}": v for k,v in stats.items()}
        })
    except Exception as exc:
        cv_rows.append({"date": date, "checkpoint": None, "error": repr(exc)})
    finally:
        del model, hedge, ctx
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

cv_audit = pd.DataFrame(cv_rows)
print(cv_audit.to_string(index=False))
cv_audit.to_csv(DIAG_DIR / "04_daily_cv_quality.csv", index=False)


      date checkpoint                                              error
2024-03-01       None TypeError('DailyMarketCurve() takes no arguments')
2024-03-04       None TypeError('DailyMarketCurve() takes no arguments')
2024-03-05       None TypeError('DailyMarketCurve() takes no arguments')
2024-03-06       None TypeError('DailyMarketCurve() takes no arguments')
2024-03-07       None TypeError('DailyMarketCurve() takes no arguments')


## 6. Next-day evaluation-universe audit

The original v7 headline next-day metric used all matched contracts in the cleaned call/put chain. The Neural-LV model, however, was calibrated on one OTM quote per strike.

This section uses the already-exported v7 contract-level next-day results and reports the same errors on:

- the entire matched clean chain;
- contracts that correspond to the v7 OTM observation on the current date;
- contracts that are the OTM observation on the next date;
- contracts that are OTM observations on **both** dates.

No new model simulation is performed.


In [7]:

transition_file = V7_CACHE_DIR / "v7_next_day_pricing_hedging_contracts.csv"
if not transition_file.exists():
    raise FileNotFoundError(
        f"{transition_file} not found. Run the completed original v7 notebook first."
    )

transitions = pd.read_csv(transition_file, low_memory=False)
transitions["date"] = pd.to_datetime(transitions["date"])
transitions["next_date"] = pd.to_datetime(transitions["next_date"])

otm_id_map = {
    pd.Timestamp(d): set(g["optionid_otm"].dropna().astype(int))
    for d, g in otm.groupby("date")
}
transitions["optionid_int"] = transitions["optionid"].astype(int)
transitions["current_is_v7_otm"] = [
    oid in otm_id_map.get(pd.Timestamp(d), set())
    for d, oid in zip(transitions["date"], transitions["optionid_int"])
]
transitions["next_is_v7_otm"] = [
    oid in otm_id_map.get(pd.Timestamp(d), set())
    for d, oid in zip(transitions["next_date"], transitions["optionid_int"])
]
transitions["both_dates_v7_otm"] = transitions["current_is_v7_otm"] & transitions["next_is_v7_otm"]

def transition_price_stats(g):
    e = g["next_day_pricing_error"].to_numpy(float)
    mid = np.maximum(g["mid_next"].to_numpy(float), 1e-12)
    out = {
        "n": len(g),
        "RMSE": float(np.sqrt(np.mean(e**2))) if len(g) else np.nan,
        "MAE": float(np.mean(np.abs(e))) if len(g) else np.nan,
        "Rel_MAE_pct": float(np.mean(np.abs(e)/mid)*100) if len(g) else np.nan,
    }
    if "next_day_inside_bid_ask" in g:
        out["inside_bid_ask"] = float(pd.to_numeric(g["next_day_inside_bid_ask"], errors="coerce").mean())
    return out

universe_rows = []
for (d, nd), g in transitions.groupby(["date", "next_date"], sort=True):
    subsets = {
        "full_clean_chain": g,
        "current_date_OTM": g.loc[g["current_is_v7_otm"]],
        "next_date_OTM": g.loc[g["next_is_v7_otm"]],
        "OTM_on_both_dates": g.loc[g["both_dates_v7_otm"]],
    }
    for label, sub in subsets.items():
        universe_rows.append({"date": d, "next_date": nd, "universe": label, **transition_price_stats(sub)})

universe_audit = pd.DataFrame(universe_rows)
print(universe_audit.to_string(index=False))

pivot_rmse = universe_audit.pivot(index=["date","next_date"], columns="universe", values="RMSE")
if {"full_clean_chain","OTM_on_both_dates"}.issubset(pivot_rmse.columns):
    pivot_rmse["full_over_both_OTM_RMSE"] = pivot_rmse["full_clean_chain"] / pivot_rmse["OTM_on_both_dates"]
    print("\nFULL-CHAIN / BOTH-DATES-OTM RMSE RATIO")
    print(pivot_rmse.to_string())

universe_audit.to_csv(DIAG_DIR / "05_next_day_universe_audit.csv", index=False)
pivot_rmse.reset_index().to_csv(DIAG_DIR / "05_next_day_universe_rmse_pivot.csv", index=False)


      date  next_date          universe    n      RMSE      MAE  Rel_MAE_pct  inside_bid_ask
2024-03-01 2024-03-04  full_clean_chain 8771  5.821381 4.456823     4.050289        0.069205
2024-03-01 2024-03-04  current_date_OTM 4200  2.194130 1.688355     6.017538        0.077143
2024-03-01 2024-03-04     next_date_OTM 4200  2.309272 1.736055     6.023085        0.075714
2024-03-01 2024-03-04 OTM_on_both_dates 3932  1.994206 1.568275     6.190307        0.079095
2024-03-04 2024-03-05  full_clean_chain 8994  5.320432 4.397700     4.880539        0.128197
2024-03-04 2024-03-05  current_date_OTM 4343  4.493041 3.648185     8.210533        0.056182
2024-03-04 2024-03-05     next_date_OTM 4343  4.101165 3.289950     7.918495        0.068156
2024-03-04 2024-03-05 OTM_on_both_dates 3951  4.229316 3.445964     8.532723        0.060744
2024-03-05 2024-03-06  full_clean_chain 9248  8.694495 6.943723     6.565846        0.030709
2024-03-05 2024-03-06  current_date_OTM 4468  4.000212 3.398589    10.

## 7. Market-delta hedge diagnostic

OptionMetrics' supplied `delta_market` is not treated as ground truth. It is used as a same-contract benchmark for the **same** one-day diagnostic:

\[
\Delta P-\Delta\,\Delta S.
\]

If both market delta and Neural-LV delta perform badly on a particular transition, the large hedge residual is likely dominated by one-day theta / volatility-surface movement rather than only by an NLV delta defect.


In [8]:

required = {"delta_market", "neural_lv_delta", "dP_market", "dS"}
missing = required - set(transitions.columns)
if missing:
    raise KeyError(f"Transition export is missing required columns: {sorted(missing)}")

transitions["market_delta_hedge_error"] = (
    transitions["dP_market"].astype(float)
    - transitions["delta_market"].astype(float) * transitions["dS"].astype(float)
)
transitions["delta_difference"] = transitions["neural_lv_delta"].astype(float) - transitions["delta_market"].astype(float)

def hedge_stats(g, error_col):
    e = g[error_col].dropna().to_numpy(float)
    return {
        "RMSE": float(np.sqrt(np.mean(e**2))) if len(e) else np.nan,
        "MAE": float(np.mean(np.abs(e))) if len(e) else np.nan,
        "bias": float(np.mean(e)) if len(e) else np.nan,
    }

delta_rows = []
for (d, nd), g0 in transitions.groupby(["date", "next_date"], sort=True):
    for universe, g in [
        ("full_clean_chain", g0),
        ("OTM_on_both_dates", g0.loc[g0["both_dates_v7_otm"]]),
    ]:
        hs_nlv = hedge_stats(g, "hedge_error")
        hs_mkt = hedge_stats(g, "market_delta_hedge_error")
        good = g[["neural_lv_delta","delta_market"]].dropna()
        dd = (good["neural_lv_delta"] - good["delta_market"]).to_numpy(float)
        corr = good["neural_lv_delta"].corr(good["delta_market"]) if len(good) > 1 else np.nan
        delta_rows.append({
            "date": d, "next_date": nd, "universe": universe, "n": len(g),
            "NLV_hedge_RMSE": hs_nlv["RMSE"], "NLV_hedge_MAE": hs_nlv["MAE"], "NLV_hedge_bias": hs_nlv["bias"],
            "market_delta_hedge_RMSE": hs_mkt["RMSE"], "market_delta_hedge_MAE": hs_mkt["MAE"],
            "market_delta_hedge_bias": hs_mkt["bias"],
            "NLV_minus_market_delta_RMSE": float(np.sqrt(np.mean(dd**2))) if len(dd) else np.nan,
            "NLV_minus_market_delta_bias": float(np.mean(dd)) if len(dd) else np.nan,
            "delta_correlation": float(corr) if pd.notna(corr) else np.nan,
        })

delta_audit = pd.DataFrame(delta_rows)
print(delta_audit.to_string(index=False))
delta_audit.to_csv(DIAG_DIR / "06_market_delta_hedge_audit.csv", index=False)


: 

: 

## 8. Same-day hold-out residual concentration

This section uses the existing v7 hold-out export. It does not change the loss or introduce bucket weights; the buckets are purely descriptive.

The purpose is to detect whether the current errors are concentrated in a particular maturity or moneyness region before introducing any mechanism intended to protect such a region.


In [8]:

holdout_file = V7_CACHE_DIR / "v7_holdout_contract_results.csv"
if not holdout_file.exists():
    raise FileNotFoundError(f"{holdout_file} not found.")

holdout_results = pd.read_csv(holdout_file, low_memory=False)
holdout_results["date"] = pd.to_datetime(holdout_results["date"])
holdout_results["abs_k"] = holdout_results["k"].abs()

# Intervals chosen only for readable diagnostics.
holdout_results["T_bucket"] = pd.cut(
    holdout_results["T"],
    bins=[0, 30/365, 90/365, 180/365, np.inf],
    labels=["7-30d", "31-90d", "91-180d", "181d+"],
    include_lowest=True,
)
holdout_results["k_bucket"] = pd.cut(
    holdout_results["abs_k"],
    bins=[0, 0.025, 0.075, 0.15, np.inf],
    labels=["|k|<=2.5%", "2.5-7.5%", "7.5-15%", "15%+"],
    include_lowest=True,
)

def error_group_table(df, group_cols):
    def one(g):
        e = g["pricing_error"].to_numpy(float)
        sse = float(np.sum(e**2))
        return pd.Series({"n": len(g), "RMSE": np.sqrt(np.mean(e**2)), "MAE": np.mean(np.abs(e)), "SSE": sse})
    out = df.groupby(group_cols, observed=True).apply(one).reset_index()
    total = out.groupby("date")["SSE"].transform("sum")
    out["share_of_date_SSE"] = out["SSE"] / total
    return out

maturity_residuals = error_group_table(holdout_results, ["date", "T_bucket"])
moneyness_residuals = error_group_table(holdout_results, ["date", "k_bucket"])

# Exact-maturity concentration: largest SSE share per date.
exact_T = error_group_table(holdout_results, ["date", "T"])
top_exact_T = (
    exact_T.sort_values(["date", "share_of_date_SSE"], ascending=[True, False])
    .groupby("date", as_index=False)
    .head(5)
)

print("\nMATURITY-BUCKET RESIDUALS")
print(maturity_residuals.to_string(index=False))
print("\nMONEYNESS-BUCKET RESIDUALS")
print(moneyness_residuals.to_string(index=False))
print("\nTOP 5 EXACT MATURITIES BY SHARE OF HOLD-OUT SSE")
print(top_exact_T.to_string(index=False))

maturity_residuals.to_csv(DIAG_DIR / "07_holdout_maturity_residuals.csv", index=False)
moneyness_residuals.to_csv(DIAG_DIR / "07_holdout_moneyness_residuals.csv", index=False)
top_exact_T.to_csv(DIAG_DIR / "07_top_exact_maturity_SSE_share.csv", index=False)



MATURITY-BUCKET RESIDUALS
      date T_bucket    n     RMSE      MAE        SSE  share_of_date_SSE
2024-03-01    7-30d 45.0 1.270281 1.104957  72.612669           0.144094
2024-03-01   31-90d 69.0 1.250717 0.900342 107.936264           0.214191
2024-03-01  91-180d 40.0 1.630229 1.020220 106.305903           0.210956
2024-03-01    181d+ 45.0 2.196308 1.476479 217.069531           0.430758
2024-03-04    7-30d 55.0 1.472649 1.176668 119.278213           0.193368
2024-03-04   31-90d 65.0 1.218518 0.874199  96.511123           0.156459
2024-03-04  91-180d 40.0 2.425970 1.317062 235.413150           0.381641
2024-03-04    181d+ 40.0 2.034962 1.324604 165.642800           0.268532
2024-03-05    7-30d 55.0 1.588402 1.250228 138.766152           0.161085
2024-03-05   31-90d 65.0 1.757486 1.374960 200.769163           0.233060
2024-03-05  91-180d 40.0 2.161777 1.446415 186.931238           0.216996
2024-03-05    181d+ 40.0 2.893883 1.913357 334.982351           0.388859
2024-03-06    7-30d 55.0

C:\Users\Startklar\AppData\Local\Temp\ipykernel_39352\2257464905.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = df.groupby(group_cols, observed=True).apply(one).reset_index()
C:\Users\Startklar\AppData\Local\Temp\ipykernel_39352\2257464905.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = df.groupby(group_cols, observed=True).apply(one).reset_index()
C:\Users\Startklar\AppData\Local\Tem

## 9. Rolling trunk-time-transfer audit

The original v7:

- copies the previous trunk parameters unchanged;
- shifts the maturity heads to approximate the same calendar expiry.

But the trunk itself receives time \(t\) as its first input. For the same calendar point after advancing the valuation date by \(\Delta t\), the old network should be queried at \(t+\Delta t\).

Because the first trunk layer is affine in \((t,x)\), the time shift can be represented **exactly** in the copied trunk before the ReLU by:

\[
b_{\mathrm{new}} = b_{\mathrm{old}} + W_t\,\Delta t.
\]

This audit compares:

- the target previous-day volatility \(\sigma_{
m old}(t+\Delta t,x)\);
- the original v7 naive transfer;
- the same transfer with this first-layer bias correction.

No weights are optimized.


In [9]:

def transfer_heads_and_trunk_naive(previous, current, previous_date, current_date):
    current.trunk.load_state_dict(previous.trunk.state_dict())
    elapsed = (pd.Timestamp(current_date) - pd.Timestamp(previous_date)).days / 365.0
    mismatches = []
    with torch.no_grad():
        for i, T_new in enumerate(current.boundaries):
            target_old_T = float(T_new) + elapsed
            j = int(np.argmin(np.abs(previous.boundaries - target_old_T)))
            current.heads[i].load_state_dict(previous.heads[j].state_dict())
            mismatches.append(abs(float(previous.boundaries[j]) - target_old_T))
    return elapsed, np.asarray(mismatches)

def apply_exact_trunk_time_shift(current, elapsed):
    # FeedForwardTrunk input order is [t, x]; the first module is nn.Linear(2, width).
    first = current.trunk.net[0]
    if not isinstance(first, nn.Linear) or first.weight.shape[1] != 2:
        raise RuntimeError("Unexpected trunk first layer; cannot apply exact time-shift diagnostic.")
    with torch.no_grad():
        first.bias.add_(first.weight[:, 0] * float(elapsed))

@torch.no_grad()
def transfer_error(previous, new_boundaries, previous_date, current_date):
    naive = MaturityHeadNeuralLocalVol(new_boundaries).to(DEVICE).eval()
    elapsed, mismatch = transfer_heads_and_trunk_naive(previous, naive, previous_date, current_date)

    shifted = MaturityHeadNeuralLocalVol(new_boundaries).to(DEVICE).eval()
    transfer_heads_and_trunk_naive(previous, shifted, previous_date, current_date)
    apply_exact_trunk_time_shift(shifted, elapsed)

    times = []
    starts = np.r_[0.0, new_boundaries[:-1]]
    for a, b in zip(starts, new_boundaries):
        if b <= a:
            continue
        for frac in np.linspace(0.2, 0.8, TRANSFER_POINTS_PER_INTERVAL):
            times.append(float(a + frac*(b-a)))

    sq_naive, sq_shifted, sq_base, n = 0.0, 0.0, 0.0, 0
    x_np = TRANSFER_X_GRID.astype(np.float32)
    x = torch.tensor(x_np[:, None], dtype=torch.float32, device=DEVICE)

    for t in times:
        old_t = t + elapsed
        target = previous(torch.full_like(x, old_t), x)
        pred_naive = naive(torch.full_like(x, t), x)
        pred_shifted = shifted(torch.full_like(x, t), x)
        sq_naive += float(torch.sum((pred_naive-target)**2).cpu())
        sq_shifted += float(torch.sum((pred_shifted-target)**2).cpu())
        sq_base += float(torch.sum(target**2).cpu())
        n += target.numel()

    rmse_naive = math.sqrt(sq_naive/n)
    rmse_shifted = math.sqrt(sq_shifted/n)
    rms_target = math.sqrt(sq_base/n)
    return {
        "elapsed_years": elapsed,
        "median_head_mismatch_years": float(np.median(mismatch)),
        "max_head_mismatch_years": float(np.max(mismatch)),
        "naive_sigma_RMSE": rmse_naive,
        "shifted_sigma_RMSE": rmse_shifted,
        "naive_relative_RMSE": rmse_naive/max(rms_target, 1e-12),
        "shifted_relative_RMSE": rmse_shifted/max(rms_target, 1e-12),
    }


# Audit transitions for which the previous day's saved model exists.
transfer_rows = []
model_dates = available_saved_dates()

for current_date in model_dates[1:]:
    previous_candidates = [d for d in model_dates if d < current_date]
    if not previous_candidates:
        continue
    previous_date = max(previous_candidates)

    prev_model = prev_ctx = None
    try:
        _, prev_train, _ = daily_split(previous_date)
        prev_ctx = build_calibration_context(previous_date, prev_train)

        # No hedge and no optimizer state is kept alive after this call.
        prev_model, _ = load_saved_model(previous_date, prev_ctx)

        _, cur_train, _ = daily_split(current_date)
        new_boundaries = np.sort(cur_train["T"].unique())
        stats = transfer_error(
            prev_model, new_boundaries, previous_date, current_date
        )
        transfer_rows.append({
            "previous_date": previous_date,
            "current_date": current_date,
            **stats
        })
    except Exception as exc:
        transfer_rows.append({
            "previous_date": previous_date,
            "current_date": current_date,
            "error": repr(exc)
        })
    finally:
        del prev_model, prev_ctx
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

transfer_audit = pd.DataFrame(transfer_rows)
print(transfer_audit.to_string(index=False))
transfer_audit.to_csv(DIAG_DIR / "08_trunk_time_transfer_audit.csv", index=False)


previous_date current_date                                              error
   2024-03-01   2024-03-04 TypeError('DailyMarketCurve() takes no arguments')
   2024-03-04   2024-03-05 TypeError('DailyMarketCurve() takes no arguments')
   2024-03-05   2024-03-06 TypeError('DailyMarketCurve() takes no arguments')
   2024-03-06   2024-03-07 TypeError('DailyMarketCurve() takes no arguments')


## 10. Compact decision summary for the v7 revision

This final cell does not make training decisions automatically. It collects the measurements most useful for deciding which existing mechanisms are empirically justified.

The generated CSVs in `neural_lv_v7_diagnostics/` retain the full detail.


In [10]:

summary = {
    "training_independent": True,
    "march_dates": int(len(density_summary)),
    "overall_selected_fraction_of_clean_OTM": float(
        density_summary["n_selected"].sum() / density_summary["n_otm_available"].sum()
    ),
    "max_train_count_CV_across_dates": float(density_summary["train_per_maturity_CV"].max()),
    "min_balanced_over_raw_contract_weight": float(effective_weights["balanced_over_raw"].min()),
    "max_balanced_over_raw_contract_weight": float(effective_weights["balanced_over_raw"].max()),
    "median_slice_fraction_retained_by_25_cap": float(cap_audit["retained_fraction"].median()),
    "median_moneyness_span_retained_by_25_cap": float(cap_audit["moneyness_span_retained"].median()),
}

if len(history_audit):
    summary.update({
        "n_saved_rolling_histories": int(len(history_audit)),
        "monitor_metric_winner_disagreement_rate": float(1.0-history_audit["same_monitor_winner"].mean()),
        "mean_raw_selection_penalty_pct_at_balanced_monitor_winner": float(history_audit["raw_selection_penalty_pct"].mean()),
        "days_with_negative_last3_raw_slope": int((history_audit["last3_raw_RMSE_slope_per_epoch"] < 0).sum()),
    })

if len(rerank_audit):
    summary.update({
        "rerank_metric_winner_disagreement_rate": float(1.0-rerank_audit["same_rerank_winner"].mean()),
        "mean_raw_selection_penalty_pct_at_balanced_rerank_winner": float(rerank_audit["raw_rerank_selection_penalty_pct"].mean()),
    })

if len(cv_audit) and "cv_ratio_median" in cv_audit:
    vals = pd.to_numeric(cv_audit["cv_ratio_median"], errors="coerce").dropna()
    if len(vals):
        summary.update({
            "daily_CV_ratio_median_of_medians": float(vals.median()),
            "daily_CV_ratio_worst_median": float(vals.max()),
        })

if len(universe_audit):
    u = universe_audit.pivot(index=["date","next_date"], columns="universe", values="RMSE")
    if {"full_clean_chain","OTM_on_both_dates"}.issubset(u.columns):
        ratio = (u["full_clean_chain"]/u["OTM_on_both_dates"]).replace([np.inf,-np.inf], np.nan).dropna()
        if len(ratio):
            summary["mean_full_chain_over_both_OTM_RMSE_ratio"] = float(ratio.mean())

if len(delta_audit):
    d0 = delta_audit.loc[delta_audit["universe"] == "OTM_on_both_dates"].copy()
    if len(d0):
        summary.update({
            "mean_OTM_NLV_hedge_RMSE": float(d0["NLV_hedge_RMSE"].mean()),
            "mean_OTM_market_delta_hedge_RMSE": float(d0["market_delta_hedge_RMSE"].mean()),
            "mean_OTM_delta_correlation": float(d0["delta_correlation"].mean()),
        })

if len(transfer_audit):
    naive = pd.to_numeric(transfer_audit.get("naive_relative_RMSE"), errors="coerce").dropna()
    shifted = pd.to_numeric(transfer_audit.get("shifted_relative_RMSE"), errors="coerce").dropna()
    if len(naive):
        summary["mean_naive_trunk_transfer_relative_RMSE"] = float(naive.mean())
    if len(shifted):
        summary["mean_time_shifted_trunk_transfer_relative_RMSE"] = float(shifted.mean())

print(json.dumps(summary, indent=2))
with open(DIAG_DIR / "diagnostic_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nSaved diagnostic outputs under:", DIAG_DIR.resolve())
print("\nSend me the printed summary plus the detailed tables above; that is enough to make the evidence-based v7 revision.")


NameError: name 'history_audit' is not defined